In [2]:
import pandas as pd
import numpy as np
import requests
from owlready2 import *
import rdflib
import re
from collections import Counter

# Importing publications

## DLS

In [2]:
# Import and concat all DLS publications
pathname = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/synchrotron-proposals-topic-classification/Datasets'
pub_dls=pd.concat([
pd.read_csv(pathname+'/DLS/DLS annual review highlight.csv'),
pd.read_csv(pathname+'/DLS/DLS book chapter.csv'),
pd.read_csv(pathname+'/DLS/DLS conference paper.csv'),
pd.read_csv(pathname+'/DLS/DLS editor note.csv'),
pd.read_csv(pathname+'/DLS/DLS journal paper (2002-2010).csv'),
pd.read_csv(pathname+'/DLS/DLS journal paper (2011-2020).csv'),
pd.read_csv(pathname+'/DLS/DLS journal paper (2021-2024).csv'),
pd.read_csv(pathname+'/DLS/DLS magazine article.csv'),
pd.read_csv(pathname+'/DLS/DLS poster.csv'),
pd.read_csv(pathname+'/DLS/DLS report.csv'),
pd.read_csv(pathname+'/DLS/DLS science highlight.csv'),
pd.read_csv(pathname+'/DLS/DLS thesis.csv'),
pd.read_csv(pathname+'/DLS/DLS all publications 2024.csv'),
pd.read_csv(pathname+'/DLS/DLS all publications 2025.csv')
],ignore_index=True)

# Add 'Facility repository' column
pub_dls['Facility repository']='DLS'

# Remove entries with no DOI and duplicates based on DOI
pub_dls = pub_dls[pub_dls['DOI'].notnull()].drop_duplicates(subset=['DOI'], keep='first')

In [3]:
pub_dls.columns

Index(['Publications Database Id', 'DOI', 'PMID', 'ISI ID', 'Title of Paper',
       'Authors', 'Industrial Partner Is Co-author?', 'Publication State',
       'Date Published', 'Diamond Proposal Number', 'Publication Type',
       'Title of Journal', 'Journal Volume', 'Journal Pages',
       'Title of Conference', 'Peer Reviewed', 'Magazine Title',
       'Uses Synchrotron, EM or Offline lab Data?', 'Data From Diamond?',
       'Beamlines', 'Additional Facilities If Data From Diamond',
       'Facility If Data Not From Diamond', 'Subject Areas', 'Technical Areas',
       'Keywords', 'Diamond Keywords', 'Discipline/Technical Tags', 'ISBN',
       'Book Chapter', 'Added On', 'Facility repository'],
      dtype='object')

In [4]:
len(pub_dls)

16186

## Use OpenAlex API to get abstract for the publications

In [5]:
# For OpenAlex API
from pyalex import Works, config
config.api_key = "BNOUFPbaqGXJvy7RCufdvx"

In [6]:
# Function to get abstract for each DOI
def doi_to_abstract(doi):
    doi = 'https://doi.org/' + doi if not doi.startswith('https://doi.org/') else doi
    try:
        work = Works()[doi]
        abstract = work['abstract']
        return abstract
    except Exception as e:
        return None

In [7]:
pub_dls['Abstract'] = pub_dls['DOI'].apply(doi_to_abstract)

In [11]:
len(pub_dls)

16186

In [32]:
len(pub_dls[pub_dls['Abstract'].isnull()])

3527

The OpenAlex API failed to return abstracts for 3527 publications.

In [ ]:
# import requests

# def doi_to_abstract2(doi):
#    url = f"http://api.semanticscholar.org/graph/v1/paper/{doi}"

#    # Define the query parameters
#    query_params = {"fields": "abstract"}

#    # Send the API request
#    response = requests.get(url, params=query_params)

#    # Check response status
#    if response.status_code == 200:
#       response_data = response.json()
#       # Process and print the response data as needed
#       return response_data['abstract']
#    else:
#       return None

In [36]:
import requests
def doi_to_abstract2(ls):
    r = requests.post(
        'https://api.semanticscholar.org/graph/v1/paper/batch',
        params={'fields': 'title,abstract,externalIds'},
        json={"ids": ls}
    )
    return r.json()

In [ ]:
batch_size = 500
pub_dls_missing_abstracts = pub_dls[pub_dls['Abstract'].isnull()].copy()
n = len(pub_dls_missing_abstracts)
for i in range(0, n, batch_size):
    # This slices the list from the current index to index + 500
    batch = pub_dls_missing_abstracts.iloc[i:i + batch_size]
    dois = batch['DOI'].tolist()
    results = doi_to_abstract2(dois)
    for paper in results:
        if paper is not None:
            doi = paper['externalIds']['DOI']
            abstract = paper['abstract']
            pub_dls.loc[pub_dls['DOI'] == doi, 'Abstract'] = abstract

In [73]:
len(pub_dls[pub_dls['Abstract'].isnull()])

2436

There are still 2436 publications with missing abstracts. Many are due to copyright reasons.

In [ ]:
# Code to export the processed publications to a CSV file. Uncomment the following lines to run it.
# Checkpoint data

# filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/checkpoint_data/Publications_DLS.csv'
# pub_dls.to_csv(filepath,index=False)

In [61]:
# Code to import the processed publications from a CSV file if needed. Uncomment the following lines to run it.
# Checkpoint data

filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/checkpoint_data/Publications_DLS.csv'
pub_dls = pd.read_csv(filepath,encoding='utf-8')

In [85]:
# Filter out publications with missing abstracts
pub_dls_filtered = pub_dls[pub_dls['Abstract'].notnull()].copy()

# Loading PaNET ontology

In [2]:
# Load the ontology

ontology_path='/Users/fdp54928/Documents/PaNET-classifier/Data/owlapi.xrdf'
onto=get_ontology(ontology_path).load()

In [3]:
# Run reasoner

with onto:
    sync_reasoner()  # Runs reasoning and updates inferred relationships

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/fdp54928/Documents/PaNET-classifier/.venv/lib/python3.13/site-packages/owlready2/hermit:/Users/fdp54928/Documents/PaNET-classifier/.venv/lib/python3.13/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/jt/qsn_d43943dbh0h9ngcrb3n80000gq/T/tmpov9d7gg2
* Owlready2 * HermiT took 5.292837142944336 seconds
* Owlready * Reparenting PaNET.PaNET1001000: {owl.ObjectProperty, owl.AsymmetricProperty, PaNET.PaNET1000000, owl.IrreflexiveProperty} => {PaNET.PaNET1000000}
* Owlready * Reparenting PaNET.PaNET1004000: {owl.ObjectProperty, owl.AsymmetricProperty, PaNET.PaNET1000000, owl.IrreflexiveProperty} => {PaNET.PaNET1000000}
* Owlready * Reparenting PaNET.PaNET1002000: {owl.ObjectProperty, owl.AsymmetricProperty, PaNET.PaNET1000000, owl.IrreflexiveProperty} => {PaNET.PaNET1000000}
* Owlready * Reparenting PaNET.PaNET1003000: {owl.ObjectProperty, owl.AsymmetricProperty, PaNET

In [21]:
onto_ls = list(onto.classes())
panet00001 = onto.search(iri='http://purl.org/pan-science/PaNET/PaNET00001')[0]
technique_terms = [cls for cls in panet00001.descendants() if cls.iri!='https://www.wikidata.org/wiki/Q133900']
technique_label = [cls.label[0].strip().lower() for cls in technique_terms]
technique_iri = [cls.iri for cls in technique_terms]
technique_altLabel = [cls.altLabel for cls in technique_terms]

print('Number of technique classes:',len(technique_iri))

Number of technique classes: 377


# Mapping technical tags to PaNET terms

In [ ]:
# Load the PaNET mapping file
pathname = '/Users/fdp54928/Documents/PaNET-classifier/Data/PaNET_mapping.xlsx'
panet_mapping = pd.read_excel(pathname)

In [ ]:
# Clean the 'Technical tags' column by stripping whitespace and converting to lowercase
panet_mapping['Technical tags'] = panet_mapping['Technical tags'].str.strip().str.lower()

In [ ]:
# Function to extract PaNET technique terms from a given row of the 'Technical tags' column

def extract_panet_terms(row, technical_tags_ls):
    row_set = set(term.strip().lower() for term in row.split(','))
    tech_set = set(term.strip().lower() for term in technical_tags_ls)
    found_techniques = list(row_set.intersection(tech_set))
    return found_techniques

In [ ]:
# Apply the function to the 'Discipline/Technical Tags' column of the filtered publications DataFrame
extracted_techniques = pub_dls_filtered['Discipline/Technical Tags'].fillna('').apply(lambda x: extract_panet_terms(x, panet_mapping['Technical tags'].to_list()))

In [87]:
# Check which technical tags in PaNET mapping are not found in publications

# Flatten the list of extracted techniques
ls = []
for term in extracted_techniques:
    ls.extend(term)
    x=set(ls)

# Set of technical tags in PaNET mapping that are not found in publications
y = set(panet_mapping['Technical tags'].to_list())
print('Technical tags in PaNET mapping not found in publications:')
x.symmetric_difference(y)

Technical tags in PaNET mapping not found in publications:


{'anomalous diffraction',
 'anomalous scattering',
 'correlative light electron microscopy (clem)',
 'cryo focused ion beam scanning electron microscopy (fibsem)',
 'diffracted x-ray tracking (dxt)',
 'fixed target serial synchrotron crystallography (ft-ssx)',
 'in situ diffraction',
 'infrared microscopy',
 'infrared nanospectroscopy imaging',
 'lipidic cubic phase serial synchrotron crystallography (lcp-ssx)',
 'magnetic x-ray tomography (mxt)',
 'molecular replacement',
 'multi wavelength anomalous diffraction (mad)',
 'nano infrared spectroscopy',
 'photo crystallography',
 'serial femtosecond crystallography (sfx)',
 'time resolved serial femtosecond crystallography (tr-sfx)',
 'ultra small angle x-ray scattering (usaxs)',
 'uv circular dichroism imaging',
 'uv synchrotron radiation circular dichroism (srcd)',
 'x-ray birefringence imaging (xbi)',
 'x-ray excited optical luminescence (xeol)'}

In [88]:
print('Number of publications with at least one extracted technique:')
extracted_techniques.apply(lambda x: len(x)!=0).value_counts()

Number of publications with at least one extracted technique:


Discipline/Technical Tags
True     11503
False     2247
Name: count, dtype: int64

Map the technical tags to PaNET IRIs

In [89]:
def map_technical_tags_to_panet_iris(row, mapping_dict):
    panet_terms = []
    for tag in row:
        panet_term = mapping_dict[tag.strip()]
        if str(panet_term) != 'nan':
            panet_terms.append(panet_term) 
    return panet_terms

In [90]:
mapping_dict = dict(zip(panet_mapping['Technical tags'], panet_mapping['PaNET IRI']))

extracted_panet_iris = extracted_techniques.apply(lambda x: map_technical_tags_to_panet_iris(x, mapping_dict))

In [91]:
def get_hierarchy_terms(panet_iri, ontology):
    excluded_iris = {
    'http://purl.org/pan-science/PaNET/PaNET00001', 
    'http://www.w3.org/2002/07/owl#Thing',
    'https://www.wikidata.org/wiki/Q133900'
}
    cls = ontology.search(iri=panet_iri)[0]
    hierarchy_terms = list(set(ancestor.iri.strip() for ancestor in cls.ancestors() if ancestor.iri.strip() not in excluded_iris))
    return hierarchy_terms

In [92]:
all_panet_iris_set = extracted_panet_iris.apply(lambda x: [term for panet_iri in x for term in get_hierarchy_terms(panet_iri, onto)])

In [93]:
all_panet_iris_set

0                                                       []
1                                                       []
2                                                       []
3        [http://purl.org/pan-science/PaNET/PaNET00300,...
5                                                       []
                               ...                        
16179                                                   []
16182    [http://purl.org/pan-science/PaNET/PaNET01028,...
16183    [http://purl.org/pan-science/PaNET/PaNET00201,...
16184    [http://purl.org/pan-science/PaNET/PaNET01028,...
16185    [http://purl.org/pan-science/PaNET/PaNET01188,...
Name: Discipline/Technical Tags, Length: 13750, dtype: object

In [59]:
panet_iri_to_label_map = {c.iri: c.label[0] for c in onto.classes() if len(c.label) > 0}

In [95]:
all_panet_labels_set = all_panet_iris_set.apply(lambda x: [panet_iri_to_label_map[iri] for iri in x])

In [96]:
pub_dls_filtered['PaNET_IRIs_set'] = all_panet_iris_set
pub_dls_filtered['PaNET_labels_set'] = all_panet_labels_set

In [97]:
df = pub_dls_filtered[pub_dls_filtered['PaNET_IRIs_set'].apply(lambda x: len(x)!=0)]

In [98]:
# Count the frequency of each PaNET label across all publications
panet_count = Counter([y for x in all_panet_iris_set for y in x])

# Filter the PaNET terms to include only those that appear more than 10 times
filtered_panet = {term: count for term, count in panet_count.items() if count > 10}

In [99]:
# Create a list of filtered PaNET IRIs and their corresponding labels (frequency > 10)
filter_panet_iri_ls = list(filtered_panet)
filter_panet_label_ls = [panet_iri_to_label_map[iri] for iri in filter_panet_iri_ls]

In [100]:
# Filter the DataFrame to include only the filtered PaNET IRIs and labels (frequency > 10)
df_filtered = df.copy()
df_filtered['PaNET_IRIs_set'] = df_filtered['PaNET_IRIs_set'].apply(lambda x: [term for term in x if term in filter_panet_iri_ls])
df_filtered['PaNET_labels_set'] = df_filtered['PaNET_labels_set'].apply(lambda x: [term for term in x if term in filter_panet_label_ls])

In [105]:
print('Number of publications with abstracts and tagged with PaNET terms that appear more than 10 times in total:', len(df_filtered))

Number of publications with abstracts and tagged with PaNET terms that appear more than 10 times in total: 11502


In [ ]:
# Code to export the processed publications to a CSV file. Uncomment the following lines to run it.
# Checkpoint data

# filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/checkpoint_data/df_filtered.csv'
# df_filtered.to_csv(filepath,index=False)

In [2]:
# Code to import the processed publications from a CSV file if needed. Uncomment the following lines to run it.
# Checkpoint data
# The list of strings in the 'PaNET_labels_set' and 'PaNET_IRIs_set' columns will be read as strings, so we need to convert them back to lists.

import ast

filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/checkpoint_data/df_filtered.csv'
df_filtered = pd.read_csv(filepath,encoding='utf-8')

df_filtered['PaNET_labels_set'] = df_filtered['PaNET_labels_set'].apply(ast.literal_eval)   # Convert the string representation of a list back into an actual list
df_filtered['PaNET_IRIs_set'] = df_filtered['PaNET_IRIs_set'].apply(ast.literal_eval)   # Convert the string representation of a list back into an actual list

# Training/Testing data

In [3]:
import numpy as np
from skmultilearn.model_selection import iterative_train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import joblib

In [ ]:
# Stack the titles and abstracts as two columns: Column 0 = Title, Column 1 = Abstract, Row i = ith Publication
# This is our input, X
dois = df_filtered['DOI']
titles = df_filtered['Title of Paper']
abstracts = df_filtered['Abstract']
X = np.column_stack((dois,titles,abstracts))

In [ ]:
# Binarize the labels, Y
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df_filtered['PaNET_IRIs_set'].to_list())

In [6]:
print('Number of labels:',len(mlb.classes_))
print('Number of publications:',len(Y))

Number of labels: 140
Number of publications: 11502


In [ ]:
# Split (iterative_train_test_split will keep the row relationship intact)
# We want to do a two-step split to create Train, Validation, and Test sets in a roughly 80-10-10 split while preserving the label distribution across all three sets, 
# which is crucial for multi-label classification tasks.

# First split: Train vs Test
X_train, Y_train, X_test, Y_test = iterative_train_test_split(X, Y, test_size=0.1)

# Second split: Train vs Validation (from the training set)
X_train, Y_train, X_eval, Y_eval = iterative_train_test_split(X_train, Y_train, test_size=0.11)  # 0.11 x 0.9 = 0.099 of the total data for evaluation, 0.801 for training

In [ ]:
# Save the Binarizer
# This includes .classes_ and internal mapping
filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/'
joblib.dump(mlb, filepath + 'binarizer.pkl')

# 6. Combine X and Y into DataFrames for storage
# This ensures that row i of X is always row i of Y
train_df = pd.concat([
    pd.DataFrame(X_train, columns = ['DOI','Title','Abstract']).reset_index(drop=True), 
    pd.DataFrame(Y_train, columns=mlb.classes_)
], axis=1)

test_df = pd.concat([
    pd.DataFrame(X_test,columns = ['DOI','Title','Abstract']).reset_index(drop=True), 
    pd.DataFrame(Y_test, columns=mlb.classes_)
], axis=1)

eval_df = pd.concat([
    pd.DataFrame(X_eval,columns = ['DOI','Title','Abstract']).reset_index(drop=True), 
    pd.DataFrame(Y_eval, columns=mlb.classes_)
], axis=1)

# 7. Save to Parquet
train_df.to_parquet(filepath + 'train_set.parquet', index=False)
test_df.to_parquet(filepath + 'test_set.parquet', index=False)
eval_df.to_parquet(filepath + 'eval_set.parquet', index=False)

# Build ancestor map

We need to build an ancestor map to calculate hF1 score during training/inference.

In [50]:
# Build ancestor map from PaNET
def build_ancestor_map(onto, panet_iris):
    excluded_iris = {
    'http://purl.org/pan-science/PaNET/PaNET00001', 
    'http://www.w3.org/2002/07/owl#Thing',
    'https://www.wikidata.org/wiki/Q133900'
    }
    ancestor_map = {}
    for iri in panet_iris:
        cls = onto.search_one(iri=iri)
        if cls is not None:
            # Get all ancestors excluding owl:Thing
            ancestors = {a.iri for a in cls.ancestors() if a.iri not in excluded_iris and a.iri in panet_iris}
        else:
            ancestors = set()
        ancestor_map[iri] = ancestors
    return ancestor_map

In [60]:
ancestor_map = build_ancestor_map(onto, mlb.classes_)

In [61]:
import pickle
filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/'
with open(filepath + 'ancestor_map.pkl', 'wb') as f:
    pickle.dump(ancestor_map, f)

In [63]:
# Pre-compute index-based ancestor sets for speed
ancestor_indices = []
for i, label in enumerate(mlb.classes_):
    indices = {j for j, a in enumerate(mlb.classes_) if a in ancestor_map[label]}
    ancestor_indices.append(frozenset(indices))


In [68]:
filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/'
with open(filepath + 'ancestor_indices.pkl', 'wb') as f:
    pickle.dump(ancestor_indices, f)

# HGCN

## Build adjacency matrix

In [8]:
# Load the Binarizer to get all labels/classes
import joblib
filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/binarizer.pkl'
mlb = joblib.load(filepath)

# Load the ancestor map
filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/ancestor_map.pkl'
ancestor_map = joblib.load(filepath)

In [63]:
# Build adjacency matrix for label hierarchy
# We have to add PaNET00001 (photon and neutron technique) as the root node, which is not included in mlb.classes_.
import scipy.sparse as sp
def build_adjacency_matrix(mlb, onto):
    mlb = list(mlb.classes_)
    mlb = ['http://purl.org/pan-science/PaNET/PaNET00001'] + mlb  # Add root node at the beginning
    class_to_idx = {cls: i for i, cls in enumerate(mlb)}
    num_labels = len(mlb)

    # Initialize a Directed Adjacency Matrix
    # Using a coordinate list (COO) format is often faster for building
    rows, cols = [], []
    
    for i, iri in enumerate(mlb):
        cls = onto.search_one(iri=iri)
        for parent in cls.is_a:
            parent_iri = parent.iri
            if parent_iri in class_to_idx:
                parent_idx = class_to_idx[parent_iri]
                rows.append(i)  # Child index
                cols.append(parent_idx)  # Parent index
    adjacency_matrix = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(num_labels, num_labels))
    return adjacency_matrix

In [64]:
adj_csr = build_adjacency_matrix(mlb, onto)

In [68]:
# Saving CSR matrix
filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/HGCN/'
sp.save_npz(filepath + 'hierarchy_adj.npz', adj_csr)

In [ ]:
# # Loading CSR matrix back later
# filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/'
# adj_loaded = sp.load_npz(filepath + 'hierarchy_adj.npz')


# HiAGM

In [ ]:
# Build taxononmy file for HiAGM
# We have to add PaNET00001 (photon and neutron technique) as the root node, which is not included in mlb.classes_.
def build_adjacency_matrix(mlb, onto):
    mlb = list(mlb.classes_)
    mlb = ['http://purl.org/pan-science/PaNET/PaNET00001'] + mlb  # Add root node at the beginning
    class_to_idx = {cls: i for i, cls in enumerate(mlb)}
    num_labels = len(mlb)


## Train/Test/Eval datasets

In [70]:
path = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/'
train_df = pd.read_parquet(path + 'train_set.parquet', engine='fastparquet')
test_df = pd.read_parquet(path + 'test_set.parquet', engine='fastparquet')
eval_df = pd.read_parquet(path + 'eval_set.parquet', engine='fastparquet')

In [71]:
train_df

,DOI,Title,Abstract,http://purl.org/pan-science/PaNET/PaNET00002,http://purl.org/pan-science/PaNET/PaNET00003,http://purl.org/pan-science/PaNET/PaNET00004,http://purl.org/pan-science/PaNET/PaNET00005,http://purl.org/pan-science/PaNET/PaNET00100,http://purl.org/pan-science/PaNET/PaNET00104,http://purl.org/pan-science/PaNET/PaNET00105,...,http://purl.org/pan-science/PaNET/PaNET01283,http://purl.org/pan-science/PaNET/PaNET01285,http://purl.org/pan-science/PaNET/PaNET01291,http://purl.org/pan-science/PaNET/PaNET01292,http://purl.org/pan-science/PaNET/PaNET01293,http://purl.org/pan-science/PaNET/PaNET01295,http://purl.org/pan-science/PaNET/PaNET01304,http://purl.org/pan-science/PaNET/PaNET01305,http://purl.org/pan-science/PaNET/PaNET01316,http://purl.org/pan-science/PaNET/PaNET01322
0,10.1180/EMU-notes.20.12,Synchrotron Radiation InfraRed microspectrosco...,Fourier-transform infrared (FTIR) spectroscopy...,1,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,10.5772/intechopen.70088,Multilayer polarizer at the energy of 50–1000 eV,Accurate evaluation of polarization states of ...,1,1,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,10.1007/978-1-61779-346-2_6,"Expression, purification and crystallization o...",Integral outer membrane proteins (OMPs) play k...,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,10.1117/12.3014643,Novel solutions for structural protections: re...,Protecting structural components and infrastru...,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,10.1109/INTERMAGShortPapers61879.2024.10576960,Spin and orbital moments of magnetic topologic...,Intrinsic magnetic topological insulators (MTI...,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9174,10.1042/BCJ20240605,"Structure, kinetics, and mechanism of Pseudomo...",The sulfosugar sulfoquinovose (SQ) is cataboli...,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9175,10.1182/blood.2024025861,Evaluating the impact of CRBN mutations on res...,Abstract Immunomodulatory drug (IMiD) resistan...,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9176,10.1038/s41467-025-56127-y,Broad substrate scope C-C oxidation in cyclodi...,Abstract Cyclic dipeptides are produced by org...,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9177,10.1038/s41375-025-02515-8,"Phase Ib study of PRT543, an oral protein argi...",[Image: see text],0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
